# Pre-Statistics: Mathematical Foundations with Python

This session builds the mathematical intuition and vocabulary you will need for the Statistics session. We will use the same tools introduced in the *Math in Python* session: `numpy`, `matplotlib`, `scipy`, and `sympy`.

**By the end of this session you will be able to:**
- Understand what a mathematical function is, and plot one
- Understand what a distribution is, and how to describe it
- Understand what integration means, and why "probability is area"
- Read and use a differential equation as a model of change

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, stats
import sympy
sympy.init_printing()

np.random.seed(42)

---
# 1. Motivation: Why does math matter in biology?

Consider two rabbit populations. Both have 100 rabbits today. One is growing quickly, the other is shrinking. A single number — a *snapshot* — cannot tell them apart. But a **function** describing how the population changes over time can.

Mathematics gives us a language to describe processes, not just states. This session is about that language.

We will use the **Lotka-Volterra predator-prey model** as our running example — a system of equations that describes how rabbit and fox populations influence each other over time:

$$\frac{dx}{dt} = \alpha x - \beta x y \qquad \text{(rabbits)}$$
$$\frac{dy}{dt} = \delta x y - \gamma y \qquad \text{(foxes)}$$

Don't worry if this looks intimidating — we will build up to it step by step.

---
# 2. Functions

A **mathematical function** takes an input and returns an output. You can think of it as a machine:

$$f(x) = x^2$$

Feed in $x = 3$, get out $9$. Feed in $x = -2$, get out $4$.

In Python, a mathematical function is just a Python function:

In [ ]:
def f(x):
    return x**2

print(f(3))
print(f(-2))

To visualize a function, we evaluate it at many points and plot the result:

In [ ]:
x = np.linspace(-3, 3, 200)

plt.plot(x, f(x))
plt.xlabel('x')
plt.ylabel('f(x) = x²')
plt.title('A simple function')


## Parameters shift and scale functions

A function can have **parameters** that control its shape. For example:

$$g(x) = a \cdot e^{-b \cdot x^2}$$

- $a$ controls the **height** (scale)
- $b$ controls the **width** (spread)

This particular shape — a symmetric hill — will come up again very soon.

In [ ]:
x = np.linspace(-5, 5, 300)

for a in [0.2, 0.5, 1.0, 2.0]:
    plt.plot(x, np.exp(-a * x**2), label=f'a={a}')

plt.xlabel('x')
plt.ylabel('g(x)=exp(-ax²)')
plt.title('Same shape, different width')
plt.legend()


### Exercise 1

Plot the function $h(x) = \sin(x) / x$ for $x \in [-10, 10]$.  
Hint: be careful about dividing by zero at $x=0$ — try using `np.sinc`.

In [ ]:
# Your code here

---
# 3. What is a Distribution?

Imagine measuring the height of 1000 people. You get 1000 numbers. That's a lot to look at — so instead, you ask: **how are these values spread out?**

A **distribution** answers that question. It tells you which values are common and which are rare.

The simplest way to visualize a distribution is a **histogram**: count how many values fall into each bin.

In [ ]:
# Simulate 1000 human heights in cm (mean=170, std=10)
heights = np.random.normal(loc=170, scale=10, size=1000)

plt.hist(heights, bins=30, edgecolor='white')
plt.xlabel('Height (cm)')
plt.ylabel('Count')
plt.title('Distribution of heights (n=1000)')


Notice the characteristic symmetric hill shape — this is the **normal distribution** (also called Gaussian distribution). Most people are near the average, fewer are at the extremes.

## Discrete vs. Continuous distributions

Some things can only take whole-number values — like the number of heads in 10 coin flips. These follow a **discrete distribution**.

Other things can take any real value — like height, weight, or temperature. These follow a **continuous distribution**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Discrete: number of heads in 10 coin flips, repeated 2000 times
coin_flips = np.random.binomial(n=10, p=0.5, size=2000)
axes[0].hist(coin_flips, bins=np.arange(-0.5, 11.5), edgecolor='white', align='mid')
axes[0].set_xlabel('Number of heads')
axes[0].set_ylabel('Count')
axes[0].set_title('Discrete: coin flips (n=10, p=0.5)')

# Continuous: heights
axes[1].hist(heights, bins=30, edgecolor='white')
axes[1].set_xlabel('Height (cm)')
axes[1].set_ylabel('Count')
axes[1].set_title('Continuous: heights')

plt.tight_layout()

## From histogram to density

If we normalize the histogram so the total area equals 1, we get a **probability density function (PDF)**. Instead of "how many values fell here", it tells us "how likely is a value near here".

We can overlay the theoretical PDF curve on top:

In [ ]:
x = np.linspace(130, 210, 300)

plt.hist(heights, bins=30, edgecolor='white', density=True, alpha=0.6, label='Data')
plt.plot(x, stats.norm.pdf(x, loc=170, scale=10), 'r-', lw=2, label='Normal PDF')
plt.xlabel('Height (cm)')
plt.ylabel('Probability density')
plt.title('Histogram vs. PDF')
plt.legend()

The PDF is just a **function** — exactly like the ones we plotted in Section 2. Its special property is that it is always non-negative, and the area under it equals 1. We will come back to this.

### Exercise 2

Simulate 500 values from a normal distribution with mean 0 and standard deviation 1 (this is called the **standard normal**).  
Plot a normalized histogram and overlay the standard normal PDF using `stats.norm.pdf`.

In [ ]:
# Your code here

---
# 4. Describing a Distribution: Mean, Variance, Standard Deviation

A full distribution contains a lot of information. Often we summarize it with a few numbers.

## Mean — where is the center?

The **mean** (average) tells you where the distribution is centered:

$$\mu = \frac{1}{n} \sum_{i=1}^n x_i$$

In [ ]:
print(f"Mean height: {np.mean(heights):.1f} cm")

## Variance and Standard Deviation — how spread out is it?

The **variance** measures average squared distance from the mean:

$$\sigma^2 = \frac{1}{n} \sum_{i=1}^n (x_i - \mu)^2$$

The **standard deviation** ($\sigma$) is the square root of variance — it's in the same units as the data, so it's more intuitive.

In [ ]:
print(f"Variance: {np.var(heights):.1f} cm²")
print(f"Standard deviation: {np.std(heights):.1f} cm")

## Visualizing what mean and std actually mean

In [ ]:
mu = np.mean(heights)
sigma = np.std(heights)

x = np.linspace(130, 210, 300)
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'k-', lw=2)

# Mark the mean
plt.axvline(mu, color='red', ls='--', label=f'Mean = {mu:.1f}')

# Shade ±1 std
x_fill = np.linspace(mu - sigma, mu + sigma, 200)
plt.fill_between(x_fill, stats.norm.pdf(x_fill, mu, sigma), alpha=0.3, label=f'±1 std ({sigma:.1f} cm)')

plt.xlabel('Height (cm)')
plt.ylabel('Density')
plt.title('Mean and standard deviation')
plt.legend()

For a normal distribution, approximately **68%** of values fall within ±1 standard deviation of the mean. This is a useful rule of thumb.

## Shifting and scaling

What happens to the distribution if we change the mean or standard deviation?

In [ ]:
x = np.linspace(100, 250, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Different means, same std
for mu in [150, 170, 190]:
    axes[0].plot(x, stats.norm.pdf(x, mu, 10), label=f'μ={mu}')
axes[0].set_title('Different means → shift')
axes[0].set_xlabel('x')
axes[0].legend()

# Same mean, different stds
for sigma in [5, 10, 20]:
    axes[1].plot(x, stats.norm.pdf(x, 170, sigma), label=f'σ={sigma}')
axes[1].set_title('Different std → spread')
axes[1].set_xlabel('x')
axes[1].legend()

plt.tight_layout()

### Exercise 3

Generate two groups of data:
- Group A: 300 samples, mean=50, std=5
- Group B: 300 samples, mean=55, std=10

Plot both as normalized histograms on the same axes.  
Can you tell them apart by eye? Which has more overlap with the other?

In [ ]:
# Your code here

---
# 5. Integration: Area Under the Curve

**Integration** is the mathematical operation that computes the area under a curve.

For a function $f(x)$, the definite integral from $a$ to $b$ is:

$$\int_a^b f(x)\, dx$$

Geometrically: it is the area between the curve and the x-axis, from $x=a$ to $x=b$.

## Why does this matter for statistics?

For a probability density function (PDF), **probability = area**.

The probability that a random value falls between $a$ and $b$ is exactly:

$$P(a \leq X \leq b) = \int_a^b f(x)\, dx$$

And since something must happen, the total area under any PDF equals 1:

$$\int_{-\infty}^{+\infty} f(x)\, dx = 1$$

Let's verify this numerically using `scipy.integrate.quad`:

In [ ]:
from scipy.integrate import quad

# Define the standard normal PDF manually
def normal_pdf(x, mu=0, sigma=1):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma)**2)

total_area, error = quad(normal_pdf, -np.inf, np.inf)
print(f"Total area under normal PDF: {total_area:.6f}")

Now let's visualize what it means to compute a probability as an area:

In [ ]:
x = np.linspace(-4, 4, 300)
pdf = normal_pdf(x)

# Probability that X is between 1 and 2
a, b = 1, 2
prob, _ = quad(normal_pdf, a, b)

plt.plot(x, pdf, 'k-', lw=2)

x_fill = np.linspace(a, b, 200)
plt.fill_between(x_fill, normal_pdf(x_fill), alpha=0.5, color='steelblue',
                 label=f'P(1 ≤ X ≤ 2) = {prob:.3f}')

ax_height = plt.gca().get_ylim()[1]
plt.axvline(a,ymin=0, ymax=normal_pdf(a)/ax_height, color='steelblue', ls='--', alpha=0.7)
plt.axvline(b, ymin=0, ymax=normal_pdf(b)/ax_height, color='steelblue', ls='--', alpha=0.7)
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Probability = Area under the PDF')
plt.legend()

### Exercise 4

Using `quad`, compute and visualize:
1. The probability that a standard normal variable is **greater than 1.96** (this is a number you will see a lot in statistics!)
2. The probability that it falls **between -1 and 1**

Shade the relevant area in each case.

In [ ]:
# Your code here

## Symbolic integration with SymPy

We can also verify the PDF property symbolically. Let's use SymPy to integrate the standard normal PDF from $-\infty$ to $+\infty$:

In [ ]:
x_sym = sympy.symbols('x')
mu_sym, sigma_sym = sympy.symbols('mu sigma', positive=True)

# Standard normal PDF
pdf_sym = (1 / (sympy.sqrt(2 * sympy.pi))) * sympy.exp(-x_sym**2 / 2)
pdf_sym

In [ ]:
sympy.integrate(pdf_sym, (x_sym, -sympy.oo, sympy.oo))

The integral equals 1 — confirmed analytically.

---
# 6. Differential Equations: Modeling Change

So far we've looked at static distributions. But many biological phenomena are **dynamic** — populations grow, diseases spread, neurons fire.

A **differential equation** describes a quantity in terms of how it *changes*, rather than what it *is*.

## A simple example: exponential growth

If a population grows at a rate proportional to its size:

$$\frac{dN}{dt} = r \cdot N$$

This says: *the rate of change of $N$ at any moment equals $r$ times the current $N$*.

The solution is $N(t) = N_0 \cdot e^{rt}$, but we don't need to solve it by hand — `scipy` can integrate it numerically:

In [ ]:
from scipy.integrate import odeint

def exponential_growth(N, t, r):
    return r * N

t = np.linspace(0, 10, 200)
N0 = 10  # initial population

for r in [0.1, 0.3, 0.5]:
    N = odeint(exponential_growth, N0, t, args=(r,))
    plt.plot(t, N, label=f'r={r}')

plt.xlabel('Time')
plt.ylabel('Population')
plt.title('Exponential growth: dN/dt = r·N')
plt.legend()

## Back to Lotka-Volterra

Now we're ready to return to our motivating example. Two coupled ODEs — one for prey (rabbits), one for predators (foxes):

$$\frac{dx}{dt} = \alpha x - \beta x y$$
$$\frac{dy}{dt} = \delta x y - \gamma y$$

- $\alpha$: rabbit birth rate
- $\beta$: rate at which foxes eat rabbits  
- $\delta$: rate at which foxes grow from eating rabbits
- $\gamma$: fox death rate

In [ ]:
params = {"alpha": 1.5, "beta": 1.0, "delta": 1.0, "gamma": 3.0}

def lotka_volterra(X, t, alpha, beta, delta, gamma):
    x, y = X
    dxdt = alpha * x - beta * x * y
    dydt = delta * x * y - gamma * y
    return [dxdt, dydt]

t = np.linspace(0, 15, 1000)
X0 = [10, 5]  # 10 rabbits, 5 foxes

solution = odeint(lotka_volterra, X0, t,
                  args=(params['alpha'], params['beta'], params['delta'], params['gamma']))

plt.plot(t, solution[:, 0], label='Rabbits')
plt.plot(t, solution[:, 1], label='Foxes')
plt.xlabel('Time')
plt.ylabel('Population')
plt.title('Lotka-Volterra predator-prey model')
plt.legend()

Both populations oscillate — when rabbits are plentiful, foxes thrive; when foxes are plentiful, rabbits decline; then foxes decline too, and rabbits recover. A self-sustaining cycle.

## Phase portrait

Instead of plotting both populations vs. time, we can plot them against each other. This is called a **phase portrait** and shows the overall behavior of the system:

In [ ]:
# Multiple trajectories from different starting conditions
fig, ax = plt.subplots(figsize=(6, 5))

for rabbits_0 in [5, 10, 15]:
    for foxes_0 in [3, 5, 7]:
        sol = odeint(lotka_volterra, [rabbits_0, foxes_0], t,
                     args=(params['alpha'], params['beta'], params['delta'], params['gamma']))
        ax.plot(sol[:, 0], sol[:, 1], alpha=0.6)

ax.set_xlabel('Rabbits')
ax.set_ylabel('Foxes')
ax.set_title('Phase portrait: Lotka-Volterra')

Each closed loop is a different trajectory — the system cycles forever without settling to a fixed point.

### Exercise 5

Find the **equilibrium** of the Lotka-Volterra system — the population sizes at which nothing changes (both derivatives equal zero).

Use SymPy to solve the system symbolically, then verify by running the ODE with those initial conditions and checking the populations stay constant.

In [ ]:
# Hint: set up symbols and use sympy.nonlinsolve
from sympy import nonlinsolve

x, y, alpha, beta, delta, gamma = sympy.symbols('x y alpha beta delta gamma', positive=True)

# Your code here

---
# 7. (Bonus) Where does the Normal Distribution come from?

The normal distribution has a precise mathematical formula:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

This is just a function — the same kind we plotted in Section 2. Let's inspect it symbolically:

In [ ]:
x_s, mu_s, sigma_s = sympy.symbols('x mu sigma', real=True)
sigma_s = sympy.Symbol('sigma', positive=True)

normal = (1 / (sigma_s * sympy.sqrt(2 * sympy.pi))) * sympy.exp(-((x_s - mu_s)**2) / (2 * sigma_s**2))
normal

We can compute its derivative — this tells us where the distribution peaks (the mean) and where it inflects (one std away):

In [ ]:
sympy.diff(normal, x_s)

Setting this to zero and solving gives $x = \mu$ — the peak is at the mean. Exactly what we expect.

The normal distribution appears so frequently in nature because of the **Central Limit Theorem**: the average of many independent random quantities tends to be normally distributed, regardless of what distribution those quantities individually follow. This is why your statistics session will rely on it so heavily.

---
# Summary

| Concept | What it is | Python tool |
|---|---|---|
| Function | Maps inputs to outputs | `numpy`, `matplotlib` |
| Distribution | Describes how values are spread | `numpy.random`, `scipy.stats` |
| Mean / Std | Center and spread of a distribution | `np.mean`, `np.std` |
| PDF | A function; probability = area under it | `scipy.stats` |
| Integration | Area under a curve | `scipy.integrate.quad` |
| ODE | Equation describing change over time | `scipy.integrate.odeint` |
| Symbolic math | Exact, analytic computation | `sympy` |

**You are now ready for the Statistics session**, which will build on all of these ideas — using distributions, areas, and the normal PDF to test hypotheses and draw conclusions from data.

---
# References

- [SciPy stats documentation](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [SciPy integrate documentation](https://docs.scipy.org/doc/scipy/reference/integrate.html)
- [SymPy tutorial](https://docs.sympy.org/latest/tutorial/index.html)
- [Lotka-Volterra tutorial](https://scipy-cookbook.readthedocs.io/items/LoktaVolterraTutorial.html)
- Think Stats by Allen Downey — free ebook on statistics with Python